# Spike Model Evaluation

Convergence diagnostics, sparsity analysis, replicate correlation,
and export of mutations DataFrame.

In [ ]:
config_path = "config/config.yaml"
profile = None
run_name = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import multidms

from notebooks._common import load_config, combine_replicate_muts

In [ ]:
cfg = load_config(config_path, profile, run_name=run_name)
spike = cfg["spike"]

output_dir = spike["output_dir"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
times_seen_threshold = spike["times_seen_threshold"]

warnings.simplefilter("ignore")
%matplotlib inline

In [ ]:
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))
model_collection = multidms.ModelCollection(models)
print(f"Loaded {len(models)} models")
print("Convergence:", model_collection.fit_models.converged.value_counts().to_dict())

## Convergence trajectories

In [ ]:
data = model_collection.convergence_trajectory_df()

p = (
    ggplot(data.reset_index().rename(columns={"index": "step"}), aes(x="step", y="objective_error_trajectory"))
    + geom_line()
    + facet_wrap("~dataset_name+fusionreg", scales="free_y", ncol=3)
    + theme(figure_size=(10, 8))
)
_ = p.draw(show=True)

In [ ]:
saveas = "convergence_all_lasso_lines"
cmap = plt.get_cmap("tab20")

fig, ax = plt.subplots(1, figsize=[6, 4])
for i, (idx, row) in enumerate(model_collection.fit_models.iterrows()):
    traj = row.model.convergence_trajectory_df
    col_name = "loss_trajectory" if "loss_trajectory" in traj.columns else "objective_error_trajectory"
    ax.plot(traj[col_name].values, label=f"{row.dataset_name} λ={row.fusionreg}", color=cmap(i / len(model_collection.fit_models)))
ax.set_xlabel("iteration")
ax.set_ylabel("loss")
ax.legend(bbox_to_anchor=(1, 1), fontsize=7)
sns.despine()
plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()

## Shift sparsity and replicate correlation

In [ ]:
chart, sparsity_df = model_collection.shift_sparsity(return_data=True, height_scalar=100)
chart

In [ ]:
chart, corr_df = model_collection.mut_param_dataset_correlation(width_scalar=200, return_data=True)
chart

## Export mutations DataFrame

In [ ]:
# Select model at chosen lasso strength
model_name = "model"
model = (
    model_collection.fit_models
    .query(f"fusionreg == {chosen_lasso_strength}")
    .iloc[0]
    .model
)

mut_df_replicates = combine_replicate_muts(
    {
        f"{fit.dataset_name}".split("-")[-1]: fit.model
        for fit in models.query(f"fusionreg == {chosen_lasso_strength}").itertuples()
    },
    predicted_func_scores=True,
    how="inner",
    times_seen_threshold=times_seen_threshold,
)

# Add sense column
mut_df_replicates["sense"] = [
    "stop" if mut == "*" else "nonsynonymous" for mut in mut_df_replicates.muts
]

mut_df_replicates.to_csv(f"{output_dir}/mutations_df.csv")
print(f"Saved {output_dir}/mutations_df.csv ({len(mut_df_replicates)} mutations)")
mut_df_replicates.head()